# 02D — Extract poses from the full Biking video (safe baseline)

The five-frame diagnostic succeeded. This notebook scales up to the **whole selected video** while keeping the known stable settings:
- CPU inference for this first full pass (diagnostic choice; it does not mean the GPU is faulty).
- Sample every 4th original frame.
- YOLO person detection + ViTPose; **no tracker and no MP4 encoding in this pass**.
- Save a JPEG overlay for each sampled frame and checkpoint the pose arrays every 5 sampled frames.

This is a baseline extraction only: it picks the largest detected person independently in each sampled frame, so person identity may change. We will add tracking only after the full-frame extraction is stable.

Run cells in order, one at a time. Do not use Run All.

## Cell 1 — Load paths, model and detector

In [1]:
from pathlib import Path
import sys, gc, time
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
sys.path.insert(0, str(PROJECT_DIR))

from src.vitpose_model import (
    ViTPose, get_affine_transform, decode_heatmaps, draw_skeleton_on,
    IMG_H, IMG_W, IMG_MEAN, IMG_STD
)

CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "epoch_209.pth"
YOLO_PATH = PROJECT_DIR / "notebooks" / "yolo11n.pt"
MANIFEST_PATH = PROJECT_DIR / "data" / "processed" / "manifest.csv"
assert CHECKPOINT_PATH.is_file(), f"Checkpoint missing: {CHECKPOINT_PATH}"
assert YOLO_PATH.is_file(), f"YOLO weights missing: {YOLO_PATH}"
assert MANIFEST_PATH.is_file(), f"Manifest missing: {MANIFEST_PATH}"

# CPU is intentionally used in this baseline to avoid repeating the earlier kernel crash.
DEVICE = torch.device("cpu")
torch.set_num_threads(2)

print("Loading ViTPose checkpoint on CPU...", flush=True)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model = ViTPose()
model.load_state_dict(checkpoint["model_state"], strict=True)
model.eval()
print("ViTPose loaded. Saved epoch:", checkpoint.get("epoch", "unknown"), flush=True)
del checkpoint
gc.collect()

print("Loading YOLO detector...", flush=True)
detector = YOLO(str(YOLO_PATH))
print("YOLO loaded. Device for this test:", DEVICE, flush=True)

Loading ViTPose checkpoint on CPU...
ViTPose loaded. Saved epoch: 209
Loading YOLO detector...
YOLO loaded. Device for this test: cpu


## Cell 2 — Select one Biking video and set outputs

Check that the selected path is correct. This chooses the first `Biking` row in the existing manifest.

In [2]:
manifest = pd.read_csv(MANIFEST_PATH)
biking = manifest[manifest["label"].astype(str).str.casefold().eq("biking")].reset_index(drop=True)
assert len(biking) > 0, "No Biking videos found in manifest."

VIDEO_PATH = Path(str(biking.loc[0, "video_path"]))
VIDEO_LABEL = str(biking.loc[0, "label"])
assert VIDEO_PATH.is_file(), f"Video not found: {VIDEO_PATH}"

FRAME_STEP = 4                 # process original frames 0, 4, 8, ...
DETECTION_CONF = 0.25
POSE_THRESH = 0.10
SAVE_PARTIAL_EVERY = 5         # save arrays every 5 sampled frames

FRAME_DIR = PROJECT_DIR / "outputs" / "pose_videos" / "biking_full_video_frames"
FRAME_DIR.mkdir(parents=True, exist_ok=True)
KEYPOINT_DIR = PROJECT_DIR / "data" / "processed" / "keypoints_single_tracked"
KEYPOINT_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_NPZ = KEYPOINT_DIR / "biking_full_video_partial.npz"
FINAL_NPZ = KEYPOINT_DIR / "biking_full_video_keypoints.npz"

print("Label:", VIDEO_LABEL)
print("Video:", VIDEO_PATH)
print("Video exists:", VIDEO_PATH.is_file())
print("Sample interval:", FRAME_STEP, "original frames")
print("JPEG output folder:", FRAME_DIR)

Label: Biking
Video: C:\Users\admin\ViTPose_Action_Recognition\data\raw\ucf_sports\Biking\v_Biking_g01_c01.avi
Video exists: True
Sample interval: 4 original frames
JPEG output folder: C:\Users\admin\ViTPose_Action_Recognition\outputs\pose_videos\biking_full_video_frames


## Cell 3 — Define the pose inference helper

In [3]:
def pose_for_box_cpu(frame_rgb, bbox):
    x1, y1, x2, y2 = map(float, bbox)
    h, w = frame_rgb.shape[:2]
    x1 = np.clip(x1, 0, w - 1)
    x2 = np.clip(x2, 0, w)
    y1 = np.clip(y1, 0, h - 1)
    y2 = np.clip(y2, 0, h)
    bw, bh = x2 - x1, y2 - y1
    if bw < 2 or bh < 2:
        return None

    center = np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)
    scale = max(bw, bh * IMG_W / IMG_H) * 1.25
    trans = get_affine_transform(center, scale, 0.0, (IMG_W, IMG_H))
    crop = cv2.warpAffine(frame_rgb, trans, (IMG_W, IMG_H))
    crop = crop.astype(np.float32) / 255.0
    crop = (crop - IMG_MEAN) / IMG_STD
    tensor = torch.from_numpy(crop.transpose(2, 0, 1).copy()).float().unsqueeze(0)

    with torch.inference_mode():
        heatmaps = model(tensor)[0].float().cpu().numpy()

    keypoints = decode_heatmaps(heatmaps, center, scale).astype(np.float32)
    del tensor, heatmaps, crop
    return keypoints


def save_pose_npz(path, keypoints_list, boxes_list, frame_indices, has_person,
                  label, video_path, fps, frame_step):
    # np.savez_compressed writes a compact copy of the observations captured so far.
    np.savez_compressed(
        path,
        keypoints=np.stack(keypoints_list).astype(np.float32),
        boxes=np.stack(boxes_list).astype(np.float32),
        frame_indices=np.asarray(frame_indices, dtype=np.int32),
        has_person=np.asarray(has_person, dtype=np.bool_),
        label=np.asarray(str(label)),
        video_path=np.asarray(str(video_path)),
        fps=np.asarray(float(fps), dtype=np.float32),
        frame_step=np.asarray(int(frame_step), dtype=np.int32),
    )

print("Pose helper is ready.", flush=True)

Pose helper is ready.


## Cell 4 — Process the entire video safely

This reads every video frame so the sampling indices stay correct, but runs detection and pose estimation only on every 4th frame. It does not keep the full video in RAM. Each sampled-frame overlay is written to disk immediately. A partial `.npz` file is refreshed every five sampled frames in case the run is interrupted.

In [4]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), f"Cannot open video: {VIDEO_PATH}"

fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
reported_total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
print("FPS:", fps, "| reported total frames:", reported_total_frames, flush=True)

keypoints_list, boxes_list, frame_indices, has_person = [], [], [], []
frame_index = 0
sampled_count = 0
person_found_count = 0
started = time.time()

try:
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        if frame_index % FRAME_STEP == 0:
            print(f"Sample {sampled_count + 1}: original frame {frame_index}", flush=True)
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            det = detector.predict(
                frame_bgr, classes=[0], conf=DETECTION_CONF,
                device="cpu", verbose=False
            )[0]

            bbox = np.full((4,), np.nan, dtype=np.float32)
            keypoints = np.full((17, 3), np.nan, dtype=np.float32)
            found = False

            if det.boxes is not None and len(det.boxes) > 0:
                boxes = det.boxes.xyxy.cpu().numpy()
                areas = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
                bbox = boxes[int(np.argmax(areas))].astype(np.float32)
                predicted = pose_for_box_cpu(frame_rgb, bbox)
                if predicted is not None and predicted.shape == (17, 3):
                    keypoints = predicted
                    found = True
                    person_found_count += 1
                    annotated_rgb = draw_skeleton_on(frame_rgb.copy(), keypoints, thresh=POSE_THRESH)
                    x1, y1, x2, y2 = map(int, bbox)
                    cv2.rectangle(annotated_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
                else:
                    annotated_rgb = frame_rgb.copy()
            else:
                annotated_rgb = frame_rgb.copy()

            out_img = FRAME_DIR / f"frame_{frame_index:06d}.jpg"
            saved = cv2.imwrite(str(out_img), cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR))
            if not saved:
                raise RuntimeError(f"Could not save image: {out_img}")

            keypoints_list.append(keypoints)
            boxes_list.append(bbox)
            frame_indices.append(frame_index)
            has_person.append(found)
            sampled_count += 1

            # Save incremental progress every few sampled frames.
            if sampled_count % SAVE_PARTIAL_EVERY == 0:
                save_pose_npz(PARTIAL_NPZ, keypoints_list, boxes_list, frame_indices,
                              has_person, VIDEO_LABEL, VIDEO_PATH, fps, FRAME_STEP)
                print(f"  checkpointed {sampled_count} sampled frames", flush=True)

            del det, frame_rgb, annotated_rgb
            gc.collect()

        frame_index += 1
finally:
    cap.release()

if not sampled_count:
    raise RuntimeError("No frames were sampled from the video.")

# Save final complete result and keep the partial file as a recovery artifact.
save_pose_npz(FINAL_NPZ, keypoints_list, boxes_list, frame_indices,
              has_person, VIDEO_LABEL, VIDEO_PATH, fps, FRAME_STEP)

elapsed = time.time() - started
print("\nFULL VIDEO PROCESSING FINISHED", flush=True)
print("Original frames read:", frame_index, flush=True)
print("Sampled frames processed:", sampled_count, flush=True)
print("Frames with a person/pose:", person_found_count, flush=True)
print("Sampling interval:", FRAME_STEP, flush=True)
print("Elapsed seconds:", round(elapsed, 1), flush=True)
print("Final pose file:", FINAL_NPZ, flush=True)
print("Annotated JPEG folder:", FRAME_DIR, flush=True)

FPS: 29.97002997002997 | reported total frames: 151
Sample 1: original frame 0
Sample 2: original frame 4
Sample 3: original frame 8
Sample 4: original frame 12
Sample 5: original frame 16
  checkpointed 5 sampled frames
Sample 6: original frame 20
Sample 7: original frame 24
Sample 8: original frame 28
Sample 9: original frame 32
Sample 10: original frame 36
  checkpointed 10 sampled frames
Sample 11: original frame 40
Sample 12: original frame 44
Sample 13: original frame 48
Sample 14: original frame 52
Sample 15: original frame 56
  checkpointed 15 sampled frames
Sample 16: original frame 60
Sample 17: original frame 64
Sample 18: original frame 68
Sample 19: original frame 72
Sample 20: original frame 76
  checkpointed 20 sampled frames
Sample 21: original frame 80
Sample 22: original frame 84
Sample 23: original frame 88
Sample 24: original frame 92
Sample 25: original frame 96
  checkpointed 25 sampled frames
Sample 26: original frame 100
Sample 27: original frame 104
Sample 28: 

## Cell 5 — Verify the saved outputs

In [5]:
print("Output JPEG count:", len(list(FRAME_DIR.glob("frame_*.jpg"))))
print("Partial NPZ exists:", PARTIAL_NPZ.is_file())
print("Final NPZ exists:", FINAL_NPZ.is_file())

if FINAL_NPZ.is_file():
    with np.load(FINAL_NPZ, allow_pickle=False) as d:
        print("keypoints:", d["keypoints"].shape, "(sampled_frames, 17, 3)")
        print("boxes:", d["boxes"].shape)
        print("frame_indices count:", len(d["frame_indices"]))
        print("frames with person:", int(d["has_person"].sum()), "/", len(d["has_person"]))
        print("first frame indices:", d["frame_indices"][:10].tolist())
        print("label:", d["label"].item())

Output JPEG count: 38
Partial NPZ exists: True
Final NPZ exists: True
keypoints: (38, 17, 3) (sampled_frames, 17, 3)
boxes: (38, 4)
frame_indices count: 38
frames with person: 23 / 38
first frame indices: [0, 4, 8, 12, 16, 20, 24, 28, 32, 36]
label: Biking


## Interpretation / limitations

- This completes **pose extraction only**, not action recognition.
- `keypoints` should have shape `(T, 17, 3)`, where `T` is the number of sampled frames.
- NaN rows indicate sampled frames where no target person/pose was detected.
- The largest person is selected separately in each sampled frame; identity consistency is not guaranteed in this no-tracker baseline.
- If this full video succeeds, we will reintroduce tracking in a separate test, then scale to a small balanced subset of action classes.
- The original video is never changed.